In [8]:
# General Imports
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os
os.chdir(r"D:\Machine Learning\Datasets")  # Use raw string to avoid escape issues

# Torch imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

# Sklearn imports
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

# Load data
brupt = pd.read_excel("bankruptcy.xlsx", sheet_name='data', index_col=0)

# Features and target
X = brupt.drop(['D', 'YR'], axis=1).values
y = brupt['D'].values  # Binary: 0 or 1

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=25, stratify=y
)

# Scale features
scaler = MinMaxScaler()
X_scl_trn = scaler.fit_transform(X_train)
X_scl_tst = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.from_numpy(X_scl_trn).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)  # Shape: (n_samples, 1)
X_test_tensor = torch.from_numpy(X_scl_tst).float()
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

# Dataset and DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)  # Better batch size

# Set seed for reproducibility
torch.manual_seed(25)

# Model definition (deeper network as in your attempt)
# model = nn.Sequential(
#     nn.Linear(in_features=X_scl_trn.shape[1], out_features=18),
#     nn.ReLU(),
#     nn.Linear(in_features=18, out_features=7),
#     nn.ReLU(),
#     nn.Linear(in_features=7, out_features=3),
#     nn.ReLU(),
#     nn.Linear(in_features=3, out_features=1)  # Output logits for binary classification
# )
class MLPClassifier(torch.nn.Module):
    def __init__(self,num_features):
        super(MLPClassifier, self).__init__()
        self.linear1 = nn.Linear(in_features=num_features, out_features=18)
        self.linear2 = nn.Linear(in_features=18, out_features=7)
        self.linear3 = nn.Linear(in_features=7, out_features=3)
        self.linear4 = nn.Linear(in_features=3, out_features=1)
        self.act1 = nn.ReLU()
    
    def forward(self,x):
        x= self.linear1(x)
        x= self.act1(x)
        x= self.linear2(x)
        x= self.act1(x)
        x= self.linear3(x)
        x= self.act1(x)
        output= self.linear4(x)
        return output
# Model summary (correct input shape)
model = MLPClassifier(num_features = X_scl_trn.shape[1])
summary(model, input_size=(X_scl_trn.shape[1],))  # e.g., (10,) if 10 features

# Loss and optimizer
criterion = nn.BCEWithLogitsLoss()  # Combines sigmoid + BCELoss → input logits, target float in [0,1]
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

# Training loop
num_epochs = 1000
for epoch in range(num_epochs):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(batch_X)              # Logits of shape (batch_size, 1)
        loss = criterion(outputs, batch_y)    # batch_y is (batch_size, 1), float
        
        # Backward pass
        loss.backward()
        optimizer.step()
    
    # Print every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.6f}')

# Evaluation on test set
model.eval()
with torch.no_grad():
    test_logits = model(X_test_tensor)                    # Shape: (n_test, 1)
    test_prob = torch.sigmoid(test_logits)                # Convert logits → probabilities
    test_prob_np = test_prob.squeeze(1).numpy()            # Shape: (n_test,)
    y_pred = (test_prob_np >= 0.5).astype(int)             # Predicted classes

# Metrics
accuracy = accuracy_score(y_test, y_pred)
logloss = log_loss(y_test, test_prob_np)

print(f"\nTest Accuracy: {accuracy:.4f}")
print(f"Test Log Loss: {logloss:.4f}")

Is CUDA available? False
Number of GPUs: 0
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 18]             450
              ReLU-2                   [-1, 18]               0
            Linear-3                    [-1, 7]             133
              ReLU-4                    [-1, 7]               0
            Linear-5                    [-1, 3]              24
              ReLU-6                    [-1, 3]               0
            Linear-7                    [-1, 1]               4
Total params: 611
Trainable params: 611
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------
Epoch [100/1000], Loss: 0.691303
Epoch [200/1000], Loss: 0.680745
Epoch